# Time-Series Random Forest with Grid Search

**Project:** Machine-learning-based recognition of production and changeover activities  
**CRISP-DM phase:** Modelling / Evaluation

## Purpose

This notebook trains a **random forest** (an ensemble of many decision trees) for `Label_03` (**1 = Production, 0 = Changeover**) on the original, time-stamped dataset, using a **time-aware** validation design that never trains on the future:

1. sort the rows chronologically by `time`;
2. chronological train/test split: the first 80 % of the time series for training, the last 20 % for testing;
3. `TimeSeriesSplit` cross-validation inside the training part (every validation fold lies *after* its training part);
4. `GridSearchCV` over the number of trees, the maximum depth and the minimum leaf size, selecting by F1-score;
5. evaluation on the test part, feature importances, and saving of the metrics and the model.

All numeric signals are used as features, **except the target, the other label columns and `time`**. The target is excluded explicitly and checked by an assertion.

## Workflow

This notebook reads the original dataset file, not the output of notebook 01. The thesis describes the model in Sections 3.2 and 4.3; notebook 06 (full data) is a variant without scaling and without the piece counter `Stuckzahl`.

> **Data:** The dataset is public (Zenodo, licence CC BY 4.0) and is **not** included in this repository. See the project README for the download link and citation. This notebook reads the original dataset file, saved as `data.csv` in the project root. Generated files are written to `../outputs/` and are not committed.

## Important limitations

- **Label definition.** `Label_03` is derived from the machine's own process state (Production = the machine is in process), and the NC/PLC signals describe that state directly. Very high scores can therefore partly reflect how the label is defined, not only a learned pattern. The feature-importance table (Section 6) shows which signals drive the predictions; check it for signals that are almost a copy of the label.
- **One machine, one laboratory.** The data comes from a single machine and three products; the scores say little about other machines.
- **Missing values.** The random forest handles missing values natively (scikit-learn 1.4 or newer); no imputation is applied.
- **`Stuckzahl` is a feature here.** The piece counter `Stuckzahl` is kept as a feature in this notebook (the other notebooks remove it). Check whether it makes the task easier than intended.
- **Earlier version.** An earlier version of this notebook accidentally kept `Label_03` among the features, which produced a meaningless perfect score. This is fixed. Results have to be re-generated by running the notebook.

## 1. Environment and configuration

Install the dependencies with `pip install -r requirements.txt` (run from the project root). All paths are relative to this notebook, which is expected to live in the `notebooks/` folder.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, cohen_kappa_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)

DATA_PATH = Path("../data.csv")     # original dataset, see the project README
OUTPUT_DIR = Path("../outputs")     # local results, not for upload

TARGET_COL = "Label_03"
TIME_COL = "time"
TEST_FRACTION = 0.20
SEED = 42

## 2. Load the data and sort it chronologically

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Download data_v1_0_1.csv from https://doi.org/10.5281/zenodo.15735480, "
        "save it as data.csv in the project root, or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

for col in (TIME_COL, TARGET_COL):
    if col not in df.columns:
        raise KeyError(f"Column {col!r} not found. A time-series approach needs a time column and a target.")

df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values(TIME_COL, kind="stable").reset_index(drop=True)
print("Time range:", df[TIME_COL].min(), "->", df[TIME_COL].max())

print("\nLabel distribution (0 = Changeover, 1 = Production):")
print(pd.DataFrame({
    "count": df[TARGET_COL].value_counts(),
    "share": df[TARGET_COL].value_counts(normalize=True).round(4),
}))

## 3. Build the features and make the chronological split

Features are all numeric columns except the target, the other label columns and the time column (and an index column `Unnamed: 0` if the file has one).

In [ ]:
non_feature_cols = [TIME_COL, "Unnamed: 0"]
other_labels = [c for c in df.columns if c.startswith("Label_") and c != TARGET_COL]
print("Other label columns excluded from the features:", other_labels)


def build_xy(frame):
    drop_cols = [c for c in non_feature_cols + other_labels + [TARGET_COL] if c in frame.columns]
    X = frame.drop(columns=drop_cols).select_dtypes(include=[np.number]).copy()
    y = frame[TARGET_COL].astype(int).to_numpy()

    # Safeguards against target leakage
    assert TARGET_COL not in X.columns, "The target must not be part of the features."
    assert not any(c.startswith("Label_") for c in X.columns), "No label column may be a feature."
    return X, y


n_total = len(df)
n_test = int(n_total * TEST_FRACTION)
train_df = df.iloc[:-n_test]
test_df = df.iloc[-n_test:]
print(f"Total rows: {n_total:,}   train: {len(train_df):,}   test: {len(test_df):,}")
print("Train period:", train_df[TIME_COL].min(), "->", train_df[TIME_COL].max())
print("Test period: ", test_df[TIME_COL].min(), "->", test_df[TIME_COL].max())

X_train, y_train = build_xy(train_df)
X_test, y_test = build_xy(test_df)
feature_names = X_train.columns.tolist()
print(f"Using {len(feature_names)} numeric features.")
print(f"Class counts, train: {np.bincount(y_train).tolist()}   test: {np.bincount(y_test).tolist()}")

## 4. Time-series cross-validation and grid search

`TimeSeriesSplit` cuts the training period into consecutive parts; in each split the model is trained on the earlier part and validated on the part that follows. The grid has 2 × 3 × 3 = 18 parameter combinations × 5 splits = 90 fits, which can take a few minutes.

The `StandardScaler` is not required for a random forest (trees are insensitive to scaling); it is kept to stay close to the original workflow.

In [ ]:
pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_jobs=-1,
        class_weight="balanced",
        random_state=SEED,
    )),
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 3, 5],
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="f1",
    n_jobs=1,          # the forest itself already uses all cores; avoids running out of memory
    verbose=1,
)

# Plain numpy arrays avoid warnings about feature names in the pipeline
X_train_np = X_train.to_numpy()
X_test_np = X_test.to_numpy()

print("Fitting GridSearchCV with TimeSeriesSplit...")
grid.fit(X_train_np, y_train)

print("Best parameters:", grid.best_params_)
print(f"Best cross-validated F1-score: {grid.best_score_:.4f}")

best_model = grid.best_estimator_

## 5. Evaluation on the chronological test set

The test set is the last 20 % of the recording, which the model has never seen. Sensitivity is the recall of the Production class; specificity is the share of Changeover rows that are recognized correctly.

In [ ]:
y_pred = best_model.predict(X_test_np)
y_proba = best_model.predict_proba(X_test_np)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

results_df = pd.DataFrame({
    "ACC": [accuracy_score(y_test, y_pred)],
    "F1": [f1_score(y_test, y_pred, zero_division=0)],
    "AUC": [roc_auc_score(y_test, y_proba)],
    "Sensitivity": [recall_score(y_test, y_pred, zero_division=0)],
    "Precision": [precision_score(y_test, y_pred, zero_division=0)],
    "Specificity": [tn / (tn + fp) if (tn + fp) else np.nan],
    "Kappa": [cohen_kappa_score(y_test, y_pred)],
    "TN": [tn], "FP": [fp], "FN": [fn], "TP": [tp],
}, index=["TimeSeries_RF"])
display(results_df.round(4))

print(classification_report(y_test, y_pred, digits=4, zero_division=0))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred, labels=[0, 1]), annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion matrix: time-series RF (GridSearch)")
plt.tight_layout()
plt.show()

## 6. Which signals matter? Feature importances

The importance of a feature is how much it reduces the impurity in the forest's trees, summed over all trees. If one or two signals dominate completely, check whether they are almost a copy of the label (see the limitations).

In [ ]:
importances = pd.Series(
    best_model.named_steps["model"].feature_importances_, index=feature_names
).sort_values(ascending=False)

display(importances.head(15).round(4).to_frame("importance"))

top15 = importances.head(15)
plt.figure(figsize=(7, 5))
plt.barh(top15.index[::-1], top15.values[::-1])
plt.xlabel("Importance")
plt.title("Top 15 feature importances: time-series RF")
plt.tight_layout()
plt.show()

## 7. Save the results locally

Written to `../outputs/` (not for upload): the metrics, the feature importances, the list of features, and the fitted model as a `joblib` file.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_out = OUTPUT_DIR / "timeseries_rf_metrics.csv"
importance_out = OUTPUT_DIR / "timeseries_rf_feature_importance.csv"
features_out = OUTPUT_DIR / "rf_timeseries_label03_features.txt"
model_out = OUTPUT_DIR / "rf_timeseries_label03.joblib"

results_df.to_csv(metrics_out)
importances.to_frame("importance").to_csv(importance_out)
features_out.write_text("\n".join(feature_names) + "\n")
joblib.dump(best_model, model_out)

for path in (metrics_out, importance_out, features_out, model_out):
    print("Saved:", path)

## 8. Summary and conclusion

This notebook evaluates a random forest with a validation design that respects the time order, which is stricter than the random splits of notebooks 02 to 05. The random forest can model non-linear rules and is much less prone to memorizing single rows than one deep tree.

No reference results are listed here: the numbers from the earlier version are invalid, because the target was among the features. Run the notebook on the downloaded dataset to obtain the real values, and compare the cross-validated F1-score (Section 4) with the test F1-score (Section 5). Then check the feature importances (Section 6) before interpreting a very high score.

**Next steps:** compare with the same model on the 40-feature subset, test on additional machines, and consider a multi-class label that separates more detailed states.